# json_repair

json-repair 0.64.9 repairs the broken JSON that language models write -- missing quotes, trailing commas, single
quotes, prose around the object, cut-off output -- and returns the Python object or a fixed JSON string. These cells
check known answers for each kind of break, the string and object forms, a truncated reply, the `schema` extra (a
pydantic model or a JSON Schema decides the types), and the `json_repair` command.

In [ ]:
# Example 1: the common breaks a language model makes, each repaired to a known object.
import json_repair

cases = {
    '{"a": 1, "b": 2,}': {"a": 1, "b": 2},
    "{'name': 'x', 'ok': True}": {"name": "x", "ok": True},
    '{name: "x"}': {"name": "x"},
    "[1, 2, 3": [1, 2, 3],
}
for broken, want in cases.items():
    got = json_repair.loads(broken)
    print(repr(broken), "->", got)
    assert got == want, (broken, got)

In [ ]:
# Example 2: repair_json returns a valid JSON string, and finds the object inside prose and a code fence.
import json
import json_repair

fixed = json_repair.repair_json('{"a": [1, 2', ensure_ascii=False)
print(fixed)
assert json.loads(fixed) == {"a": [1, 2]}

fenced = json_repair.repair_json('Here is the JSON: ```json\n{"x": 1}\n``` thanks')
print(fenced)
assert json.loads(fenced) == {"x": 1}

In [ ]:
# Example 3: a reply cut off mid-string keeps everything before the cut.
import json_repair

truncated = '{"items": [{"id": 1, "label": "alpha"}, {"id": 2, "label": "bet'
repaired = json_repair.loads(truncated)
print(repaired)
assert repaired["items"][0] == {"id": 1, "label": "alpha"}
assert repaired["items"][1]["id"] == 2 and repaired["items"][1]["label"].startswith("bet")

In [ ]:
# Example 4: the schema extra -- a pydantic model or its JSON Schema decides the types of the repaired object.
import json_repair
import pydantic


class Item(pydantic.BaseModel):
    id: int
    tags: list[str]


by_model = json_repair.loads('{"id": "7", "tags": "x"}', schema=Item)
by_json_schema = json_repair.loads('{"id": 7, "tags": ["a", "b",]}', schema=Item.model_json_schema())
print(by_model, by_json_schema)
assert by_model == {"id": 7, "tags": ["x"]}
assert by_json_schema == {"id": 7, "tags": ["a", "b"]}

In [ ]:
# Example 5: the json_repair command repairs a file and prints valid JSON.
import json
import os
import shutil
import subprocess
import tempfile

with tempfile.TemporaryDirectory() as folder:
    path = os.path.join(folder, "broken.json")
    with open(path, "w") as handle:
        handle.write("{'k': [1, 2,],}")
    out = subprocess.run([shutil.which("json_repair"), path], capture_output=True, text=True, check=True).stdout
print(out)
assert json.loads(out) == {"k": [1, 2]}